In [4]:
"""
01_eda.py
---------
Simple exploratory data analysis (EDA) on the Netflix Titles dataset.

What this does, step by step:
1. Load netflix_titles.csv into a pandas DataFrame.
2. Print basic info (shape, missing values).
3. Look at genre trends (the "listed_in" column).
4. Look at country trends (the "country" column).
5. Look at release year trends.
6. Save simple seaborn charts for each into an "outputs" folder.

HOW TO RUN:
Put this script and "netflix_titles.csv" in the SAME folder, then run:
    python 01_eda.py
(or open it as a notebook cell-by-cell — just keep the CSV next to the notebook)
"""

import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # lets plots save to file without needing a display
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------------
# STEP 1: Setup — simple, no fancy path tricks. Just keep the CSV next to
# this script/notebook and everything below works.
# ---------------------------------------------------------------------------
DATA_FILE = "netflix_titles.csv"
OUTPUT_FOLDER = "outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

sns.set_style("whitegrid")

# ---------------------------------------------------------------------------
# STEP 2: Load the data
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_FILE)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Rows x Columns: {df.shape}")
print()
print("Movies vs TV Shows:")
print(df["type"].value_counts())
print()
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print()

# ---------------------------------------------------------------------------
# STEP 3: Genre trends
# "listed_in" holds multiple genres per title, e.g. "Dramas, International Movies"
# We split on the comma so each genre gets counted separately.
# ---------------------------------------------------------------------------
all_genres = []
for genre_string in df["listed_in"].dropna():
    for genre in genre_string.split(","):
        all_genres.append(genre.strip())

genre_counts = pd.Series(all_genres).value_counts()

print("Top 15 genres:")
print(genre_counts.head(15))
print()

plt.figure(figsize=(10, 6))
top15_genres = genre_counts.head(15).sort_values()
sns.barplot(x=top15_genres.values, y=top15_genres.index, color="#B81D24")
plt.title("Top 15 Netflix Genres by Title Count")
plt.xlabel("Number of Titles")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "top_genres.png"), dpi=150)
plt.close()

# ---------------------------------------------------------------------------
# STEP 4: Country trends
# "country" can also list multiple countries (co-productions), so we split
# the same way.
# ---------------------------------------------------------------------------
all_countries = []
for country_string in df["country"].dropna():
    for country in country_string.split(","):
        all_countries.append(country.strip())

country_counts = pd.Series(all_countries).value_counts()

print("Top 10 countries:")
print(country_counts.head(10))
print()

plt.figure(figsize=(10, 6))
top10_countries = country_counts.head(10).sort_values()
sns.barplot(x=top10_countries.values, y=top10_countries.index, color="#221F1F")
plt.title("Top 10 Countries by Netflix Title Count")
plt.xlabel("Number of Titles")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "top_countries.png"), dpi=150)
plt.close()

# ---------------------------------------------------------------------------
# STEP 5: Release year trends
# ---------------------------------------------------------------------------
titles_per_year = df["release_year"].value_counts().sort_index()
recent_years = titles_per_year[titles_per_year.index >= 1990]

print("Titles released per year (1990+, last 15 years):")
print(recent_years.tail(15))
print()

plt.figure(figsize=(10, 5))
plt.plot(recent_years.index, recent_years.values, marker="o", color="#B81D24")
plt.title("Netflix Catalog: Titles by Release Year (1990+)")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "titles_by_release_year.png"), dpi=150)
plt.close()

print(f"Done! Charts saved in the '{OUTPUT_FOLDER}' folder.")

DATASET OVERVIEW
Rows x Columns: (7787, 12)

Movies vs TV Shows:
type
Movie      5377
TV Show    2410
Name: count, dtype: int64

Missing values per column:
director        2389
cast             718
country          507
date_added        10
rating             7
show_id            0
type               0
title              0
release_year       0
duration           0
listed_in          0
description        0
dtype: int64

Top 15 genres:
International Movies        2437
Dramas                      2106
Comedies                    1471
International TV Shows      1199
Documentaries                786
Action & Adventure           721
TV Dramas                    704
Independent Movies           673
Children & Family Movies     532
Romantic Movies              531
TV Comedies                  525
Thrillers                    491
Crime TV Shows               427
Kids' TV                     414
Docuseries                   353
Name: count, dtype: int64

Top 10 countries:
United States     3297


In [2]:
"""
02_feature_engineering.py
--------------------------
Turns the messy multi-value columns (listed_in, cast, country) into clean,
structured features we can actually use for modeling and analysis.

What this does, step by step:
1. Load netflix_titles.csv.
2. Genres ("listed_in"): pick a single "primary_genre" (the first genre
   listed) + create one binary column per top genre (multi-hot encoding).
3. Countries ("country"): same idea — "primary_country" + binary columns
   for the top countries.
4. Cast ("cast"): count how many cast members are listed, and create
   binary "has_<actor>" columns for the most frequently appearing actors.
5. Save the result as netflix_titles_features.csv.

HOW TO RUN:
Put this script and "netflix_titles.csv" in the SAME folder, then run:
    python 02_feature_engineering.py
"""

import pandas as pd

DATA_FILE = "netflix_titles.csv"
OUTPUT_FILE = "netflix_titles_features.csv"

TOP_N_GENRES = 15
TOP_N_COUNTRIES = 10
TOP_N_ACTORS = 20


def split_and_strip(value: str) -> list:
    """Turn 'Dramas, International Movies' into ['Dramas', 'International Movies']."""
    if pd.isna(value):
        return []
    return [piece.strip() for piece in value.split(",")]


# ---------------------------------------------------------------------------
# STEP 1: Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_FILE)

# Break each multi-value column into a real Python list first — this list
# form is what steps 2-4 all build on.
df["genre_list"] = df["listed_in"].apply(split_and_strip)
df["country_list"] = df["country"].apply(split_and_strip)
df["cast_list"] = df["cast"].apply(split_and_strip)

# ---------------------------------------------------------------------------
# STEP 2: Genre features
# ---------------------------------------------------------------------------
df["primary_genre"] = df["genre_list"].apply(lambda genres: genres[0] if genres else "Unknown")

top_genres = (
    df["genre_list"].explode().dropna().value_counts().head(TOP_N_GENRES).index
)
for genre in top_genres:
    column_name = "genre_" + genre.replace(" ", "_").replace("&", "and")
    df[column_name] = df["genre_list"].apply(lambda genres: 1 if genre in genres else 0)

# ---------------------------------------------------------------------------
# STEP 3: Country features
# ---------------------------------------------------------------------------
df["primary_country"] = df["country_list"].apply(lambda countries: countries[0] if countries else "Unknown")

top_countries = (
    df["country_list"].explode().dropna().value_counts().head(TOP_N_COUNTRIES).index
)
for country in top_countries:
    column_name = "country_" + country.replace(" ", "_")
    df[column_name] = df["country_list"].apply(lambda countries: 1 if country in countries else 0)

# ---------------------------------------------------------------------------
# STEP 4: Cast features
# ---------------------------------------------------------------------------
df["cast_count"] = df["cast_list"].apply(len)

top_actors = (
    df["cast_list"].explode().dropna().value_counts().head(TOP_N_ACTORS).index
)
for actor in top_actors:
    column_name = "actor_" + actor.replace(" ", "_").replace(".", "")
    df[column_name] = df["cast_list"].apply(lambda cast: 1 if actor in cast else 0)

# ---------------------------------------------------------------------------
# STEP 5: Save the engineered dataset
# ---------------------------------------------------------------------------
df.to_csv(OUTPUT_FILE, index=False)

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"Rows: {len(df)}")
print(f"New genre columns: {len(top_genres)}")
print(f"New country columns: {len(top_countries)}")
print(f"New actor columns: {len(top_actors)}")
print()
print("Primary genre breakdown (top 10):")
print(df["primary_genre"].value_counts().head(10))
print()
print(f"Saved engineered dataset to: {OUTPUT_FILE}")

FEATURE ENGINEERING SUMMARY
Rows: 7787
New genre columns: 15
New country columns: 10
New actor columns: 20

Primary genre breakdown (top 10):
primary_genre
Dramas                      1384
Comedies                    1074
Documentaries                751
Action & Adventure           721
International TV Shows       690
Children & Family Movies     502
Crime TV Shows               369
Kids' TV                     359
Stand-Up Comedy              321
Horror Movies                244
Name: count, dtype: int64

Saved engineered dataset to: netflix_titles_features.csv


In [5]:
"""
03_genre_classifier.py
-----------------------
Trains a Bidirectional LSTM (TensorFlow/Keras) that reads a title's
"description" (its synopsis) and predicts its primary genre.

What this does, step by step:
1. Load netflix_titles.csv.
2. Get each title's primary genre (the first genre in "listed_in").
3. Keep only the TOP_N_GENRES most common genres — this keeps the problem
   balanced enough to learn, and gives us a clear "always guess the most
   common genre" baseline to beat.
4. Turn the description text into padded number sequences (tokenize).
5. Split into train/test sets.
6. Build an Embedding -> Bidirectional LSTM -> pooling -> Dense model.
7. Train it, then report test accuracy vs. the baseline.

HOW TO RUN:
Put this script and "netflix_titles.csv" in the SAME folder, then run:
    python 03_genre_classifier.py
This can take a few minutes on a laptop CPU — that's normal, LSTMs aren't fast.
"""

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, regularizers

# ---------------------------------------------------------------------------
# Settings — tweak these if you want to experiment
# ---------------------------------------------------------------------------
DATA_FILE = "netflix_titles.csv"
TOP_N_GENRES = 10          # only classify the 10 most common genres
VOCAB_SIZE = 8000          # how many unique words to keep
MAX_LENGTH = 60            # max words per description (longer ones get cut)
EMBEDDING_DIM = 100
LSTM_UNITS = 64
EPOCHS = 25
BATCH_SIZE = 32
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ---------------------------------------------------------------------------
# STEP 1-2: Load data and get each title's primary genre
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_FILE)
df["primary_genre"] = df["listed_in"].apply(lambda g: g.split(",")[0].strip())

# ---------------------------------------------------------------------------
# STEP 3: Keep only the top N genres
# ---------------------------------------------------------------------------
top_genres = df["primary_genre"].value_counts().head(TOP_N_GENRES).index
df = df[df["primary_genre"].isin(top_genres)].copy()

print(f"Rows kept after filtering to top {TOP_N_GENRES} genres: {len(df)}")

baseline_accuracy = df["primary_genre"].value_counts().iloc[0] / len(df)
print(f"Baseline accuracy (always guess most common genre): {baseline_accuracy:.1%}")
print()

# ---------------------------------------------------------------------------
# STEP 4: Encode text and labels
# ---------------------------------------------------------------------------
texts = df["description"].astype(str).values
labels = df["primary_genre"].values

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=MAX_LENGTH, padding="post", truncating="post")

# ---------------------------------------------------------------------------
# STEP 5: Train / test split (stratified)
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    encoded_labels,
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=encoded_labels,
)

# ---------------------------------------------------------------------------
# STEP 6: Build the Bidirectional LSTM model
# We combine average-pooling and max-pooling over the LSTM outputs — this
# tends to work better for short-text classification than either alone,
# since it captures both the overall tone and the single strongest signal.
# ---------------------------------------------------------------------------
num_classes = len(label_encoder.classes_)

inputs = layers.Input(shape=(MAX_LENGTH,))
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)(inputs)
x = layers.SpatialDropout1D(0.3)(x)
x = layers.Bidirectional(layers.LSTM(LSTM_UNITS, return_sequences=True, recurrent_dropout=0.1))(x)
avg_pool = layers.GlobalAveragePooling1D()(x)
max_pool = layers.GlobalMaxPooling1D()(x)
x = layers.Concatenate()([avg_pool, max_pool])
x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

# ---------------------------------------------------------------------------
# STEP 7: Train and evaluate
# ---------------------------------------------------------------------------
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=2,
)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print()
print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Baseline accuracy:        {baseline_accuracy:.1%}")
print(f"BiLSTM test accuracy:     {test_accuracy:.1%}")
print(f"Improvement over baseline: {test_accuracy / baseline_accuracy:.2f}x")

ImportError: dlopen(/opt/anaconda3/lib/python3.13/site-packages/tensorflow/python/platform/_pywrap_cpu_feature_guard.so, 0x0002): Symbol not found: __ZN6google8protobuf8internal15ThreadSafeArena12thread_cacheEv
  Referenced from: <5C872A48-3756-3425-9B2E-F3EB7AC5A090> /opt/anaconda3/lib/python3.13/site-packages/tensorflow/libtensorflow_framework.2.dylib
  Expected in:     <E89F2B41-0FE0-395A-8B9D-19CAC4490719> /opt/anaconda3/lib/libprotobuf.33.5.0.dylib

In [6]:
"""
01_eda.py
---------
Simple exploratory data analysis (EDA) on the Netflix Titles dataset.

What this does, step by step:
1. Load netflix_titles.csv into a pandas DataFrame.
2. Print basic info (shape, missing values).
3. Look at genre trends (the "listed_in" column).
4. Look at country trends (the "country" column).
5. Look at release year trends.
6. Save simple seaborn charts for each into an "outputs" folder.

HOW TO RUN:
Put this script and "netflix_titles.csv" in the SAME folder, then run:
    python 01_eda.py
(or open it as a notebook cell-by-cell — just keep the CSV next to the notebook)
"""

import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # lets plots save to file without needing a display
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------------
# STEP 1: Setup — simple, no fancy path tricks. Just keep the CSV next to
# this script/notebook and everything below works.
# ---------------------------------------------------------------------------
DATA_FILE = "netflix_titles.csv"
OUTPUT_FOLDER = "outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

sns.set_style("whitegrid")

# ---------------------------------------------------------------------------
# STEP 2: Load the data
# ---------------------------------------------------------------------------
df = pd.read_csv(DATA_FILE)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Rows x Columns: {df.shape}")
print()
print("Movies vs TV Shows:")
print(df["type"].value_counts())
print()
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print()

# ---------------------------------------------------------------------------
# STEP 3: Genre trends
# "listed_in" holds multiple genres per title, e.g. "Dramas, International Movies"
# We split on the comma so each genre gets counted separately.
# ---------------------------------------------------------------------------
all_genres = []
for genre_string in df["listed_in"].dropna():
    for genre in genre_string.split(","):
        all_genres.append(genre.strip())

genre_counts = pd.Series(all_genres).value_counts()

print("Top 15 genres:")
print(genre_counts.head(15))
print()

plt.figure(figsize=(10, 6))
top15_genres = genre_counts.head(15).sort_values()
sns.barplot(x=top15_genres.values, y=top15_genres.index, color="#B81D24")
plt.title("Top 15 Netflix Genres by Title Count")
plt.xlabel("Number of Titles")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "top_genres.png"), dpi=150)
plt.close()

# ---------------------------------------------------------------------------
# STEP 4: Country trends
# "country" can also list multiple countries (co-productions), so we split
# the same way.
# ---------------------------------------------------------------------------
all_countries = []
for country_string in df["country"].dropna():
    for country in country_string.split(","):
        all_countries.append(country.strip())

country_counts = pd.Series(all_countries).value_counts()

print("Top 10 countries:")
print(country_counts.head(10))
print()

plt.figure(figsize=(10, 6))
top10_countries = country_counts.head(10).sort_values()
sns.barplot(x=top10_countries.values, y=top10_countries.index, color="#221F1F")
plt.title("Top 10 Countries by Netflix Title Count")
plt.xlabel("Number of Titles")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "top_countries.png"), dpi=150)
plt.close()

# ---------------------------------------------------------------------------
# STEP 5: Release year trends
# ---------------------------------------------------------------------------
titles_per_year = df["release_year"].value_counts().sort_index()
recent_years = titles_per_year[titles_per_year.index >= 1990]

print("Titles released per year (1990+, last 15 years):")
print(recent_years.tail(15))
print()

plt.figure(figsize=(10, 5))
plt.plot(recent_years.index, recent_years.values, marker="o", color="#B81D24")
plt.title("Netflix Catalog: Titles by Release Year (1990+)")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "titles_by_release_year.png"), dpi=150)
plt.close()

print(f"Done! Charts saved in the '{OUTPUT_FOLDER}' folder.")

DATASET OVERVIEW
Rows x Columns: (7787, 12)

Movies vs TV Shows:
type
Movie      5377
TV Show    2410
Name: count, dtype: int64

Missing values per column:
director        2389
cast             718
country          507
date_added        10
rating             7
show_id            0
type               0
title              0
release_year       0
duration           0
listed_in          0
description        0
dtype: int64

Top 15 genres:
International Movies        2437
Dramas                      2106
Comedies                    1471
International TV Shows      1199
Documentaries                786
Action & Adventure           721
TV Dramas                    704
Independent Movies           673
Children & Family Movies     532
Romantic Movies              531
TV Comedies                  525
Thrillers                    491
Crime TV Shows               427
Kids' TV                     414
Docuseries                   353
Name: count, dtype: int64

Top 10 countries:
United States     3297
